### Day 5 Assignment: PySpark DataFrame Transformations & Git Integration

### Basic Tasks

#### 1. Load e-commerce dataset

In [0]:
df = spark.read.csv("/Volumes/dev/demo/ex-volume/messy_ecommerce_sales.csv", header=True, inferSchema=True)
df.display()

In [0]:
from pyspark.sql.functions import col
# 1. Rows containing NULLs
df.filter(
    col("sale_id").isNull() |
    col("customer_id").isNull() |
    col("product_id").isNull() |
    col("quantity").isNull() |
    col("sale_amount").isNull() |
    col("sale_date").isNull() |
    col("region").isNull()
).show()

In [0]:
# 2. Compare total and distinct rows
print("Total rows:", df.count())
print("Distinct rows:", df.distinct().count())

In [0]:
# 3. Remove duplicate rows
df_clean = df.dropDuplicates()
print("Rows after removing duplicates:", df_clean.count())

#### 2. Rename columns

In [0]:
df_renamed_cols = df_clean.withColumnRenamed("sale_id", "sale_ID")\
.withColumnRenamed("customer_id", "customer_ID")\
.withColumnRenamed("product_id", "product_ID")
df_renamed_cols.display()

#### 3. Connect Databricks Repo to Git 


Created the initial e-commerce data cleaning notebook, committed the changes, and pushed them to the Git repository with the commit message: "Add initial e-commerce data cleaning notebook".

###  Intermediate Tasks

####4. Build a cleaning pipeline

In [0]:
df_cleaned = df_renamed_cols.dropna(how="all").dropDuplicates().fillna({"quantity": 0, "sale_amount": 0, "region": "Unknown"}).sort("sale_date")
df_cleaned.display()

#### 5. Aggregation and JOIN

In [0]:
#Sample region dataset
region_data = [
    ("North", "Northern Region"),
    ("South", "Southern Region"),
    ("East", "Eastern Region"),
    ("West", "Western Region"),
    ("Unknown", "Unknown Region")
]

region_df = spark.createDataFrame(
    region_data,
    ["region", "region_name"]
)

display(region_df)

In [0]:
revenue_by_region = df_cleaned.groupBy("region")\
    .agg({"sale_amount": "sum"})\
    .withColumnRenamed("sum(sale_amount)", "total_revenue")\
    .join(region_df, on ="region", how = "left")
revenue_by_region.display()

#### 6. Create a feature branch